**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 33: 프로젝트 데이터 파이프라인 구축

## 🎯 데이터 파이프라인 구축

이 노트북에서는 프로젝트에 필요한 **데이터 파이프라인 전체**를 구축합니다.

### 파이프라인 흐름
```
데이터 수집 → 정제 → 형식 변환 → 합성 데이터 생성 → 검증 → Train/Eval 분할
```

### 사전 준비
- `32_project_planning.ipynb`에서 저장한 `project_plan.json` 필요
- OpenAI API 키 (합성 데이터 생성 시)
- HuggingFace 토큰 (데이터셋 다운로드 시)

In [1]:
# =====================================================
# 📦 라이브러리 임포트 + 프로젝트 계획 로드
# =====================================================
import json, os, time, requests
from dotenv import load_dotenv
load_dotenv("../.env")  # OPENAI_API_KEY 로드

OUTPUT_DIR = "./output/project"

with open(os.path.join(OUTPUT_DIR, "project_plan.json"), "r", encoding="utf-8") as f:
    project_plan = json.load(f)

# OpenAI API key 확인 → 없으면 Ollama 사용
USE_OPENAI = False
try:
    from openai import OpenAI
    client = OpenAI()
    # key 유효성 테스트
    client.models.list()
    USE_OPENAI = True
    print("✅ OpenAI API 사용 (gpt-4o-mini)")
except Exception:
    print("⚠️ OpenAI API key가 없거나 유효하지 않습니다.")
    # Ollama 확인
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            print("✅ Ollama 사용 (로컬 모델)")
        else:
            print("❌ Ollama도 실행 중이 아닙니다. ollama serve를 실행하세요.")
    except:
        print("❌ Ollama도 실행 중이 아닙니다. ollama serve를 실행하세요.")

print(f"\n📋 프로젝트: {project_plan['project_name']}")
print(f"📌 도메인: {project_plan['domain']}")
print(f"📌 생성할 데이터: {project_plan['data_config']['num_samples']}건")


✅ OpenAI API 사용 (gpt-4o-mini)

📋 프로젝트: AI/ML 기술 Q&A sLLM
📌 도메인: AI/ML 기술
📌 생성할 데이터: 50건


In [2]:
# =====================================================
# 🤖 학습 데이터 자동 생성 (OpenAI 또는 Ollama)
# =====================================================
domain = project_plan["domain"]
num_samples = project_plan["data_config"]["num_samples"]
seed_topics = project_plan["data_config"]["seed_topics"]
samples_per_topic = num_samples // len(seed_topics)

print(f"🚀 {domain} 도메인 데이터 {num_samples}건 생성 시작!")
print(f"   엔진: {'OpenAI gpt-4o-mini' if USE_OPENAI else 'Ollama (로컬)'}")
print(f"   토픽 {len(seed_topics)}개 x 토픽당 {samples_per_topic}건")
print("="*60)

def generate_with_openai(topic, n):
    """OpenAI API로 데이터 생성"""
    prompt = f"""당신은 '{domain}' 분야의 교육 데이터를 생성하는 전문가입니다.

주제: {topic}

이 주제에 대해 {n}개의 instruction-output 쌍을 JSON 배열로 생성하세요.

규칙:
- instruction: 학습자가 물어볼 법한 자연스러운 한국어 질문
- output: 200~400자 분량의 정확하고 상세한 한국어 답변
- 각 쌍은 서로 다른 관점의 질문이어야 함

반드시 아래 JSON 형식으로만 답변하세요:
[{{"instruction": "질문", "input": "", "output": "답변"}}]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    if isinstance(result, list):
        return result
    elif isinstance(result, dict):
        return list(result.values())[0] if result else []
    return []

def generate_with_ollama(topic, n):
    """Ollama로 데이터 생성"""
    prompt = f"""당신은 '{domain}' 분야의 교육 데이터를 생성하는 전문가입니다.

주제: {topic}

이 주제에 대해 {n}개의 질문-답변 쌍을 만들어주세요.
각 답변은 200~400자로 정확하고 상세하게 작성하세요.

반드시 아래 JSON 배열 형식으로만 답변하세요. 다른 텍스트 없이 JSON만 출력하세요:
[{{"instruction": "질문", "input": "", "output": "답변"}}]"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:1.5b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.8, "num_predict": 2048},
        },
        timeout=120,
    )
    
    if response.status_code != 200:
        return []
    
    text = response.json().get("response", "")
    
    # JSON 추출 시도
    try:
        # 전체가 JSON인 경우
        return json.loads(text)
    except:
        pass
    
    try:
        # JSON 부분만 추출
        start = text.find("[")
        end = text.rfind("]") + 1
        if start >= 0 and end > start:
            return json.loads(text[start:end])
    except:
        pass
    
    return []

# 데이터 생성 실행
generated_data = []
generate_fn = generate_with_openai if USE_OPENAI else generate_with_ollama

for topic_idx, topic in enumerate(seed_topics):
    print(f"\n⏳ [{topic_idx+1}/{len(seed_topics)}] 토픽: {topic}")
    
    try:
        items = generate_fn(topic, samples_per_topic)
        
        for item in items:
            if item.get("instruction") and item.get("output"):
                generated_data.append({
                    "instruction": item["instruction"],
                    "input": item.get("input", ""),
                    "output": item["output"],
                })
        
        print(f"   ✅ {len(items)}건 생성 (누적: {len(generated_data)}건)")
        time.sleep(1)
        
    except Exception as e:
        print(f"   ❌ 에러: {e}")

print(f"\n{'='*60}")
print(f"✅ 데이터 생성 완료: 총 {len(generated_data)}건")

if len(generated_data) < num_samples:
    print(f"⚠️ 목표({num_samples}건)보다 적게 생성됨. 토픽을 추가하거나 다시 실행하세요.")


🚀 AI/ML 기술 도메인 데이터 50건 생성 시작!
   엔진: OpenAI gpt-4o-mini
   토픽 10개 x 토픽당 5건

⏳ [1/10] 토픽: LoRA 파인튜닝의 원리와 장점
   ✅ 5건 생성 (누적: 5건)

⏳ [2/10] 토픽: GPU 메모리 최적화 방법
   ✅ 5건 생성 (누적: 10건)

⏳ [3/10] 토픽: 트랜스포머의 Attention 메커니즘
   ✅ 5건 생성 (누적: 15건)

⏳ [4/10] 토픽: Python 프로그래밍 기초
   ✅ 5건 생성 (누적: 20건)

⏳ [5/10] 토픽: 딥러닝 학습 기법 (옵티마이저, 스케줄러)
   ✅ 5건 생성 (누적: 25건)

⏳ [6/10] 토픽: LLM 환각 문제와 해결 방법
   ✅ 5건 생성 (누적: 30건)

⏳ [7/10] 토픽: RAG와 벡터 데이터베이스
   ✅ 5건 생성 (누적: 35건)

⏳ [8/10] 토픽: 모델 양자화와 서빙
   ✅ 5건 생성 (누적: 40건)

⏳ [9/10] 토픽: 데이터 전처리와 증강
   ✅ 5건 생성 (누적: 45건)

⏳ [10/10] 토픽: 강화학습(DPO, GRPO) 개념
   ✅ 5건 생성 (누적: 50건)

✅ 데이터 생성 완료: 총 50건


---
## 1️⃣ 데이터 수집

도메인에 맞는 데이터를 수집합니다. 아래 방법 중 해당하는 것을 선택하세요.

### 수집 방법 선택
| 방법 | 장점 | 단점 | 적합한 경우 |
|------|------|------|------------|
| HuggingFace 데이터셋 | 즉시 사용 가능, 정제됨 | 도메인 제한 | 공개 데이터 있을 때 |
| 웹 크롤링 | 최신 데이터 | 노이즈 많음, 법적 이슈 | 특수 도메인 |
| 합성 데이터 (API) | 품질 제어 가능 | 비용 발생 | 데이터 부족 시 |
| 기존 데이터 변환 | 비용 없음 | 형식 변환 필요 | 자체 데이터 있을 때 |

In [3]:
# =====================================================
# 📊 생성된 데이터 확인
# =====================================================
print(f"📊 생성된 데이터: {len(generated_data)}건")
print("="*60)

for i, item in enumerate(generated_data[:5]):
    print(f"\n{i+1}. Q: {item['instruction'][:60]}...")
    print(f"   A: {item['output'][:80]}...")

print(f"\n... (총 {len(generated_data)}건)")


📊 생성된 데이터: 50건

1. Q: LoRA 파인튜닝이란 무엇인가요?...
   A: LoRA(LoRA: Low-Rank Adaptation) 파인튜닝은 대규모 언어 모델과 같은 복잡한 모델을 효율적으로 조정하기 위한 기술입니다....

2. Q: LoRA 파인튜닝의 주요 장점은 무엇인가요?...
   A: LoRA 파인튜닝의 주요 장점 중 하나는 효율성입니다. 전통적인 파인튜닝은 모델의 모든 파라미터를 업데이트해야 하므로 많은 메모리와 시간이 소요...

3. Q: LoRA 파인튜닝을 적용할 수 있는 상황은 어떤 것들이 있나요?...
   A: LoRA 파인튜닝은 다양한 상황에서 유용하게 적용될 수 있습니다. 예를 들어, 자원이 제한된 환경에서 대규모 언어 모델을 조정해야 할 때, Lo...

4. Q: LoRA 파인튜닝과 전통적인 파인튜닝의 차이점은 무엇인가요?...
   A: LoRA 파인튜닝과 전통적인 파인튜닝은 접근 방식에서 큰 차이를 보입니다. 전통적인 파인튜닝은 전체 모델의 모든 파라미터를 업데이트하는 반면, ...

5. Q: LoRA 파인튜닝을 하기 위해 필요한 사전 지식이나 기술은 무엇인가요?...
   A: LoRA 파인튜닝을 수행하기 위해서는 기본적인 머신러닝 및 딥러닝에 대한 이해가 필요합니다. 특히, 신경망 구조와 파라미터 조정에 대한 지식이 ...

... (총 50건)


---
## 2️⃣ 데이터 정제 파이프라인

수집한 원시 데이터를 정제합니다.

### 정제 단계
1. **중복 제거**: 완전 일치 / 유사도 기반
2. **길이 필터링**: 너무 짧거나 긴 데이터 제거
3. **품질 필터링**: 의미 없는 데이터 제거
4. **언어 필터링**: 한국어 비율 확인
5. **특수문자 정리**: HTML 태그, 불필요한 기호 제거

In [4]:
# =====================================================
# 🧹 데이터 정제
# =====================================================
print("🧹 데이터 정제 중...")

# 중복 제거
seen = set()
cleaned_data = []
for item in generated_data:
    key = item["instruction"]
    if key not in seen:
        seen.add(key)
        cleaned_data.append(item)

print(f"   중복 제거: {len(generated_data)} → {len(cleaned_data)}건")

# 길이 필터링 (답변 10자 이상)
cleaned_data = [item for item in cleaned_data if len(item["output"]) >= 10]
print(f"   길이 필터: → {len(cleaned_data)}건")

print(f"✅ 정제 완료: {len(cleaned_data)}건")


🧹 데이터 정제 중...
   중복 제거: 50 → 50건
   길이 필터: → 50건
✅ 정제 완료: 50건


In [5]:
# =====================================================
# 🔄 ChatML 형식으로 변환
# =====================================================
system_prompt = project_plan["data_config"]["system_prompt"]

formatted_data = []
for item in cleaned_data:
    instruction = item["instruction"]
    input_text = item.get("input", "")
    output_text = item["output"]
    
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    
    formatted_data.append({
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output_text}
        ]
    })

print(f"✅ ChatML 변환 완료: {len(formatted_data)}건")
print(f"\n--- 변환 예시 ---")
for msg in formatted_data[0]["messages"]:
    print(f"  [{msg['role']}] {msg['content'][:80]}...")


✅ ChatML 변환 완료: 50건

--- 변환 예시 ---
  [system] 당신은 AI/ML 분야의 전문가입니다. 정확하고 이해하기 쉽게 답변해주세요....
  [user] LoRA 파인튜닝이란 무엇인가요?...
  [assistant] LoRA(LoRA: Low-Rank Adaptation) 파인튜닝은 대규모 언어 모델과 같은 복잡한 모델을 효율적으로 조정하기 위한 기술입니다....


---
## 3️⃣ 데이터 형식 변환

정제된 데이터를 학습에 적합한 형식으로 변환합니다.

### 지원 형식

**Alpaca 형식** (Instruction Tuning 표준)
```json
{"instruction": "질문", "input": "추가 컨텍스트", "output": "답변"}
```

**ShareGPT 형식** (대화형)
```json
{"conversations": [{"from": "human", "value": "질문"}, {"from": "gpt", "value": "답변"}]}
```

In [6]:
# =====================================================
# 📊 데이터 통계 분석
# =====================================================
print("📊 데이터 통계")
print("="*60)

lengths = [len(item["messages"][2]["content"]) for item in formatted_data]
print(f"   총 데이터: {len(formatted_data)}건")
print(f"   평균 답변 길이: {sum(lengths)/len(lengths):.0f}자")
print(f"   최소/최대: {min(lengths)}/{max(lengths)}자")

for i, item in enumerate(formatted_data[:3]):
    print(f"\n  {i+1}. {item['messages'][1]['content'][:50]}... ({len(item['messages'][2]['content'])}자)")


📊 데이터 통계
   총 데이터: 50건
   평균 답변 길이: 255자
   최소/최대: 185/361자

  1. LoRA 파인튜닝이란 무엇인가요?... (244자)

  2. LoRA 파인튜닝의 주요 장점은 무엇인가요?... (217자)

  3. LoRA 파인튜닝을 적용할 수 있는 상황은 어떤 것들이 있나요?... (233자)


---
## 4️⃣ 합성 데이터 생성 (OpenAI API 활용)

데이터가 부족한 경우 **OpenAI API를 활용하여 합성 데이터를 생성**합니다.

### 합성 데이터 생성 전략
- **Seed 기반 생성**: 기존 데이터를 참고하여 유사한 새 데이터 생성
- **시나리오 기반 생성**: 도메인 시나리오를 정의하고 Q&A 쌍 생성
- **Distillation**: 큰 모델의 출력을 작은 모델의 학습 데이터로 활용

> 💡 **Hint**: `gpt-4o-mini`를 사용하면 비용을 절약하면서도 좋은 품질의 데이터를 생성할 수 있습니다.

---
## 5️⃣ 데이터 검증 및 통계

최종 데이터의 품질을 검증하고 통계를 확인합니다.

In [7]:
# =====================================================
# ✅ 데이터 품질 검증
# =====================================================
print("✅ 품질 검증")
print("="*60)

errors = 0
for i, item in enumerate(formatted_data):
    msgs = item["messages"]
    if len(msgs) != 3:
        print(f"  ❌ {i}: 메시지 수 {len(msgs)}")
        errors += 1
    if not msgs[2]["content"].strip():
        print(f"  ❌ {i}: 빈 답변")
        errors += 1

if errors == 0:
    print("  ✅ 모든 데이터 정상!")
else:
    print(f"  ⚠️ {errors}건 문제 발견")


✅ 품질 검증
  ✅ 모든 데이터 정상!


In [8]:
# =====================================================
# 📊 답변 품질 샘플 확인
# =====================================================
import random
random.seed(42)

samples = random.sample(formatted_data, min(3, len(formatted_data)))
for i, item in enumerate(samples):
    print(f"\n{'='*60}")
    print(f"📌 샘플 {i+1}")
    print(f"Q: {item['messages'][1]['content']}")
    print(f"A: {item['messages'][2]['content'][:300]}")



📌 샘플 1
Q: 데이터 전처리란 무엇인가요?
A: 데이터 전처리는 머신러닝 모델을 훈련하기 위해 데이터를 준비하는 과정입니다. 이 과정에는 데이터 정제, 결측값 처리, 이상치 제거, 데이터 변환 등이 포함됩니다. 정제 과정에서 노이즈가 포함된 데이터를 제거하고, 결측값이 있는 경우에는 평균, 중위수 또는 특정 값으로 대체할 수 있습니다. 데이터 변환 단계에서는 범주형 변수를 수치형으로 변환하거나 정규화, 표준화를 통해 데이터의 분포를 일정하게 만들어 학습 성능을 향상시킬 수 있습니다.

📌 샘플 2
Q: GPU 메모리 최적화와 관련된 도구나 라이브러리는 무엇이 있나요?
A: GPU 메모리 최적화를 위해 사용할 수 있는 도구와 라이브러리는 다양합니다. NVIDIA의 'Nsight Systems'와 'Nsight Compute'는 GPU 성능을 분석하고 최적화하는 데 유용한 도구입니다. TensorFlow에서는 'tf.profiler'를 통해 메모리 사용 패턴을 분석할 수 있으며, PyTorch에서는 'torch.utils.bottleneck'을 이용해 성능 병목 현상을 파악할 수 있습니다. 또한, 메모리 사용을 줄이기 위해 'Mixed Precision Training'을 지원하는 라이브러리도 활용할 수

📌 샘플 3
Q: LoRA 파인튜닝의 주요 장점은 무엇인가요?
A: LoRA 파인튜닝의 주요 장점 중 하나는 효율성입니다. 전통적인 파인튜닝은 모델의 모든 파라미터를 업데이트해야 하므로 많은 메모리와 시간이 소요됩니다. 그러나 LoRA는 저차원 행렬을 사용해 필요한 파라미터만 업데이트하기 때문에 훨씬 적은 자원으로도 유사한 성능을 달성할 수 있습니다. 또한, LoRA는 모델의 전이 학습 능력을 높여주며, 다양한 작업에 쉽게 적응할 수 있도록 도와줍니다.


---
## 6️⃣ Train/Eval 분할

검증된 데이터를 **학습용(Train)**과 **평가용(Eval)**으로 분할합니다.

In [9]:
# =====================================================
# 📊 Train/Val 분할 및 저장
# =====================================================
import random
random.seed(42)
random.shuffle(formatted_data)

split = int(len(formatted_data) * 0.8)
train_data = formatted_data[:split]
val_data = formatted_data[split:]

# 저장
train_path = os.path.join(OUTPUT_DIR, "train_data.json")
val_path = os.path.join(OUTPUT_DIR, "val_data.json")

with open(train_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
with open(val_path, "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

# 원본 데이터도 저장 (재사용 가능)
raw_path = os.path.join(OUTPUT_DIR, "raw_data.json")
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

print(f"✅ 데이터 저장 완료!")
print(f"   학습: {len(train_data)}건 → {train_path}")
print(f"   검증: {len(val_data)}건 → {val_path}")
print(f"   원본: {len(cleaned_data)}건 → {raw_path}")
print(f"\n🎉 다음 단계: 34번 노트북 (학습)")


✅ 데이터 저장 완료!
   학습: 40건 → ./output/project/train_data.json
   검증: 10건 → ./output/project/val_data.json
   원본: 50건 → ./output/project/raw_data.json

🎉 다음 단계: 34번 노트북 (학습)


---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 데이터 수집 (HuggingFace / 크롤링 / 로컬 파일)
- ✅ 데이터 정제 (중복 제거, 길이 필터, 한국어 필터)
- ✅ 데이터 형식 변환 (Alpaca / ShareGPT)
- ✅ 합성 데이터 생성 (OpenAI API)
- ✅ 데이터 검증 및 통계 분석
- ✅ Train/Eval 분할 및 저장

### 산출물
- 📁 `my_project/data/train.json` - 학습용 데이터
- 📁 `my_project/data/eval.json` - 평가용 데이터
- 📁 `my_project/data/all_data.json` - 전체 데이터

### 다음 세션 준비사항
- 📌 데이터 파일이 올바르게 저장되었는지 확인
- 📌 데이터 수가 최소 300개 이상인지 확인
- 📌 데이터 형식이 올바른지 JSON 구조 확인

### 다음 노트북
👉 **34_project_training.ipynb**: 프로젝트 모델 학습 수행